In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Configure plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Load Episode Step Logs and Monitor Data

`episode_steps_log.tsv` records transition data per step (stationary technical features and portfolio state),
while `monitor.csv` records episode-level statistics (`r`: total reward, `l`: length, `is_success`: portfolio > initial balance).

In [ ]:
# Load step-level transition log
df = pd.read_csv('../logs/episode_steps_log.tsv', sep='\t')

# Load episode-level monitor summary (contains true terminal is_success preserved before SB3 auto-reset)
try:
    monitor_df = pd.read_csv('../logs/monitor.csv', skiprows=1)
    monitor_df['episode'] = monitor_df.index
except Exception:
    monitor_df = None

print(f"Total transition steps: {len(df):,}")
print(f"Total episodes: {df['episode'].nunique():,}")
df.head()

## 2. Inspect New Logged Features

The logged observation space includes:
- **Exogenous Stationary Signals:** 10-step trailing-Z normalized log returns (`orig_norm_log_ret*`), zero-centered RSI (`orig_rsi_centered`), Bollinger Band %B (`orig_bb_pct_b`), and moving average distances (`orig_dist_sma10`, `orig_dist_sma50`, `orig_sma_cross_ratio`).
- **Endogenous Portfolio Features:** `orig_position` ($p_t \in \{0, 1\}$), `orig_unrealized_pnl`, `orig_rel_to_last_sell`, and `orig_holding_steps_norm`.
- **Critic State Value / Q-Values:** `policy_q` (for MaskablePPO, policy state value $V(s_t)$) and `target_q`.

In [ ]:
portfolio_cols = [
    'orig_position',
    'orig_unrealized_pnl',
    'orig_rel_to_last_sell',
    'orig_holding_steps_norm',
    'policy_q',
    'reward'
]
df[portfolio_cols].describe()

## 3. Aggregate Episode Statistics

Compute cumulative rewards, trade action frequencies, and merge episode success from `monitor.csv`.

In [ ]:
# Cumulative reward per step within each episode
df['cumulative_reward'] = df.groupby('episode')['reward'].cumsum()

# Mark the last step of each episode
max_steps = df.groupby('episode')['step'].transform('max')
df['is_last_step'] = df['step'] == max_steps

# Extract terminal records per episode
episode_df = df[df['is_last_step']].copy()

# Merge monitor stats if available
if monitor_df is not None:
    episode_df = episode_df.merge(
        monitor_df[['episode', 'is_success', 'r', 'l']],
        on='episode',
        how='left'
    )
else:
    # Fallback if monitor.csv not present
    episode_df['is_success'] = episode_df['cumulative_reward'] > 0

# Compute rolling averages across episodes
for window in [10, 20, 50, 100]:
    episode_df[f'rolling_reward_{window}'] = episode_df['cumulative_reward'].rolling(window=window).mean()
    episode_df[f'rolling_success_{window}'] = episode_df['is_success'].astype(float).rolling(window=window).mean()

episode_df[['episode', 'cumulative_reward', 'is_success', 'rolling_reward_50', 'rolling_success_50']].tail()

## 4. Training Convergence and Reward Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot 1: Episode Return and Rolling Mean
ax1.plot(episode_df['episode'], episode_df['cumulative_reward'], alpha=0.3, color='steelblue', label='Episode Return')
if 'rolling_reward_50' in episode_df:
    ax1.plot(episode_df['episode'], episode_df['rolling_reward_50'], color='navy', linewidth=2, label='Rolling Mean (50 ep)')
ax1.set_title('Episode Reward / Return over Training', fontsize=14)
ax1.set_ylabel('Cumulative Reward')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Success Rate (Portfolio > Initial Balance)
if 'rolling_success_50' in episode_df:
    ax2.plot(episode_df['episode'], episode_df['rolling_success_50'], color='forestgreen', linewidth=2, label='Success Rate (50 ep rolling)')
ax2.set_title('Success Rate (Final Portfolio > Initial Balance)', fontsize=14)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Success Rate')
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Endogenous Portfolio Diagnostics & Action Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Action distribution
action_counts = df['action'].value_counts().sort_index()
action_names = ['Hold (0)', 'Buy (1)', 'Sell (2)']
axes[0, 0].bar(action_names[:len(action_counts)], action_counts.values, color=['gray', 'green', 'red'], alpha=0.7)
axes[0, 0].set_title('Action Distribution (All Steps)')
axes[0, 0].set_ylabel('Frequency')
for i, v in enumerate(action_counts.values):
    axes[0, 0].text(i, v + len(df)*0.01, f"{v:,} ({v/len(df)*100:.1f}%)", ha='center')

# 2. Unrealized PnL distribution when in position
in_pos = df[df['orig_position'] > 0.5]
axes[0, 1].hist(in_pos['orig_unrealized_pnl'], bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0, 1].set_title('Unrealized PnL (When Holding Stock)')
axes[0, 1].set_xlabel('Unrealized PnL')
axes[0, 1].set_ylabel('Count')

# 3. Normalized Holding Steps distribution
axes[1, 0].hist(in_pos['orig_holding_steps_norm'], bins=50, color='darkorange', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Normalized Holding Duration (When In Position)')
axes[1, 0].set_xlabel('Holding Steps / T_max')
axes[1, 0].set_ylabel('Count')

# 4. Critic Estimated Value vs Step Reward
sample_df = df.sample(n=min(5000, len(df)), random_state=42)
axes[1, 1].scatter(sample_df['policy_q'], sample_df['reward'], alpha=0.3, s=15, color='teal')
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1, 1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1, 1].set_title('Critic State Value V(s) vs Step Reward (Sampled)')
axes[1, 1].set_xlabel('Critic Value V(s)')
axes[1, 1].set_ylabel('Immediate Step Reward')

plt.tight_layout()
plt.show()